# 01 · Cohort and targets

This notebook reads the PM100 job table and builds the population the whole study uses:

1. the **reference cohort**: jobs that start on or after 2020-05-06, end by 2020-10-12 23:00 UTC,
   run at least 20 minutes and are not `CANCELLED` (decision P1 in [`DECISOES.md`](../DECISOES.md));
2. the **GPU cohort**: the same jobs with at least one GPU allocated;
3. the two **targets**: (a) `FAILED` ∪ `OUT_OF_MEMORY` ∪ `NODE_FAIL` against `COMPLETED`, and
   (b) `TIMEOUT` against `COMPLETED`.

Output: `data/jobs.parquet` (the reference cohort, with a `gpu` flag and the two targets).

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from tcc.config import DATA, END, MIN_RUNTIME_S, PM100_JOBS, START, STATES_A

## The PM100 job table

PM100 (Antici et al., 2023) keeps only the Marconi100 jobs that ran **alone** on all their nodes
for their whole execution, so a node's telemetry belongs to a single job. It covers the period in
which Slurm recorded the requested resources, May to October 2020. This is why the cohort, and
later the main test set, is short.

In [2]:
cols = ["job_id", "user_id", "job_state", "submit_time", "start_time", "end_time", "run_time",
        "time_limit", "qos", "partition", "mem_req", "num_nodes_req", "num_nodes_alloc",
        "num_cores_req", "num_cores_alloc", "num_gpus_req", "num_gpus_alloc", "num_tasks",
        "cores_per_task", "nodes"]
pm100 = pd.read_parquet(PM100_JOBS, columns=cols)
print(f"PM100 jobs: {len(pm100):,}")
print("first start:", pm100.start_time.min(), "  last end:", pm100.end_time.max())

PM100 jobs: 231,238


first start: 2020-05-05 15:56:00+00:00   last end: 2020-10-13 05:21:55+00:00


### Final states

In [3]:
share = pm100.job_state.value_counts(normalize=True)
print((100 * share).round(2).to_string())
print(f"\nOUT_OF_MEMORY + NODE_FAIL: {100 * (share['OUT_OF_MEMORY'] + share['NODE_FAIL']):.2f} %")

job_state
COMPLETED        77.98
FAILED           12.90
CANCELLED         4.85
TIMEOUT           3.76
OUT_OF_MEMORY     0.44
NODE_FAIL         0.07

OUT_OF_MEMORY + NODE_FAIL: 0.51 %


`TIMEOUT` marks a job that reached its time limit; the system caps it at 24 hours. `time_limit`
is in minutes:

In [4]:
print(f"largest time_limit: {pm100.time_limit.max()} min = {pm100.time_limit.max() / 60:.0f} h")

largest time_limit: 1440 min = 24 h


## Reference cohort

The limits are those of decision P1. The start time `t0` is also stored in milliseconds since the
epoch, the unit of the ExaData timestamps. Jobs are kept sorted by end time and job id, the order
the temporal splits use.

In [5]:
runtime_s = (pm100.end_time - pm100.start_time).dt.total_seconds()
keep = ((pm100.start_time >= pd.Timestamp(START, tz="UTC"))
        & (pm100.end_time <= pd.Timestamp(END, tz="UTC"))
        & (pm100.job_state != "CANCELLED")
        & (runtime_s >= MIN_RUNTIME_S))
jobs = pm100[keep].copy()
jobs["runtime_s"] = runtime_s[keep].astype("int64")
jobs["t0_ms"] = jobs.start_time.values.astype("datetime64[ms]").astype("int64")
jobs["qos"] = jobs["qos"].astype(str)
jobs = jobs.sort_values(["end_time", "job_id"]).reset_index(drop=True)
assert not jobs.job_id.duplicated().any()
assert (jobs.end_time - jobs.start_time).dt.total_seconds().eq(jobs.run_time).all(), "run_time disagrees with the times"

print(jobs.job_state.value_counts().to_string())
print(f"\nreference cohort: {len(jobs):,} jobs, {jobs.user_id.nunique()} users")

job_state
COMPLETED        52967
TIMEOUT           7905
FAILED            7663
OUT_OF_MEMORY      418
NODE_FAIL          143

reference cohort: 69,096 jobs, 392 users


## GPU filter

A job is a GPU job if at least one GPU was **allocated** (`num_gpus_alloc > 0`).

In [6]:
jobs["gpu"] = jobs.num_gpus_alloc > 0
removed = (~jobs.gpu).sum()
gpu = jobs[jobs.gpu]
print(f"removed: {removed:,} ({removed / len(jobs):.1%}); with num_gpus_req > 0 instead: "
      f"{(jobs.num_gpus_req == 0).sum():,} would be removed")
print(f"GPU cohort: {len(gpu):,} jobs, {gpu.user_id.nunique()} users")

removed: 3,079 (4.5%); with num_gpus_req > 0 instead: 3,367 would be removed
GPU cohort: 66,017 jobs, 345 users


Every node of Marconi100 has four V100 GPUs, and the data agree: a job never has more than four
GPUs per node.

In [7]:
print((gpu.num_gpus_alloc / gpu.num_nodes_alloc).value_counts().sort_index().to_string())

1.0     2085
2.0      758
3.0     1337
4.0    61837


## Targets

Each target uses only its own states: the other states get no label (NaN), so a `TIMEOUT` job is
not a negative of target (a). Prevalence is computed within each target's states.

In [8]:
s = jobs.job_state
jobs["target_a"] = np.where(s.isin(STATES_A), 1.0, np.where(s == "COMPLETED", 0.0, np.nan))
jobs["target_b"] = np.where(s == "TIMEOUT", 1.0, np.where(s == "COMPLETED", 0.0, np.nan))

for name, cohort in (("GPU cohort", jobs[jobs.gpu]), ("reference cohort", jobs)):
    for t in "ab":
        y = cohort[f"target_{t}"].dropna()
        print(f"{name}, target ({t}): {int(y.sum()):,} positives, prevalence {y.mean():.1%}")

GPU cohort, target (a): 7,987 positives, prevalence 13.6%
GPU cohort, target (b): 7,455 positives, prevalence 12.8%
reference cohort, target (a): 8,224 positives, prevalence 13.4%
reference cohort, target (b): 7,905 positives, prevalence 13.0%


## Save

In [9]:
jobs.to_parquet(DATA / "jobs.parquet", index=False)
print(f"data/jobs.parquet: {len(jobs):,} jobs ({jobs.gpu.sum():,} GPU)")

data/jobs.parquet: 69,096 jobs (66,017 GPU)
